# Дообучение RuBERT-tiny для классификации спам-сообщений

Ноутбук выполняет дообучение модели `cointegrated/rubert-tiny` (12M параметров, ультра-лёгкая модель) на собранном датасете.

Особенности обучения:
- **FP16** — смешанная точность для ускорения обучения на GPU
- **FocalLoss** — функция потерь, фокусирующаяся на сложных примерах
- **EarlyStopping** — ранняя остановка при отсутствии улучшения F1

Обучение выполняется на трёх вариантах текста:
- исходный текст (raw)
- нормализованный текст (normalized)
- предобработанный текст (preprocessed)

Фильтрация: остаются только сообщения, содержащие преимущественно кириллицу.

## Импорт необходимых библиотек

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    pipeline,
)
from datasets import Dataset

try:
    _project_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
except NameError:
    _cwd = os.getcwd()
    _project_root = _cwd
    while _project_root != '/' and not os.path.isdir(os.path.join(_project_root, 'src')):
        _project_root = os.path.dirname(_project_root)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from src.models.transformer import (
    FocalLossTrainer,
    tokenize_function,
    compute_metrics,
    is_mostly_cyrillic,
    get_training_args,
    get_model_config,
    prepare_text_variants,
    benchmark_cpu_inference,
)
from src.config import PROCESSED_DIR, MODELS_DIR

## Чтение обработанного датасета

Загружается `preprocessed.csv` из `data/processed/`.

In [2]:
df = pd.read_csv(PROCESSED_DIR / 'preprocessed.csv', index_col=0)

## Подготовка вариантов текста

Из датасета выделяются три варианта текста для дообучения:
- **raw** — исходный текст (колонка `text`)
- **normalized** — Unicode-нормализация и удаление HTML-тегов
- **preprocessed** — полная предобработка (колонка `text_preprocessed`)

Фильтрация по кириллице: остаются только сообщения, где доля кириллических символов превышает 30%.

In [3]:
variants = prepare_text_variants(df)

ru_variants = {}
for name, vdf in variants.items():
    mask = vdf['text'].apply(lambda t: is_mostly_cyrillic(str(t)))
    ru_variants[name] = vdf[mask].reset_index(drop=True)
    print(f'{name}: {len(ru_variants[name])} сообщений')

raw: 72342 сообщений
normalized: 72355 сообщений
preprocessed: 72503 сообщений


## Дообучение RuBERT-tiny

Модель: `cointegrated/rubert-tiny`.
Количество классов: 2 (ham=0, spam=1).

### Загрузка модели и токенайзера

In [4]:
MODEL_NAME = 'cointegrated/rubert-tiny'
model_config = get_model_config(MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

### Подготовка датасетов и токенизация

Для каждого варианта текста создаётся отдельный train/test сплит (80/20) и токенизируется.

In [5]:
tokenized = {}
for name, vdf in ru_variants.items():
    ds = Dataset.from_pandas(vdf)
    split = ds.train_test_split(test_size=0.2, seed=42)
    train_ds = split['train']
    test_ds = split['test']

    train_tok = train_ds.map(
        tokenize_function,
        batched=True,
        fn_kwargs={'tokenizer': tokenizer, 'max_length': model_config['max_length']},
    )
    test_tok = test_ds.map(
        tokenize_function,
        batched=True,
        fn_kwargs={'tokenizer': tokenizer, 'max_length': model_config['max_length']},
    )

    tokenized[name] = {'train': train_tok, 'test': test_tok}
    print(f'{name}: train={{len(train_tok)}}, test={{len(test_tok)}}')

Map:   0%|          | 0/57873 [00:00<?, ? examples/s]

Map:   0%|          | 0/14469 [00:00<?, ? examples/s]

raw: train={len(train_tok)}, test={len(test_tok)}


Map:   0%|          | 0/57884 [00:00<?, ? examples/s]

Map:   0%|          | 0/14471 [00:00<?, ? examples/s]

normalized: train={len(train_tok)}, test={len(test_tok)}


Map:   0%|          | 0/58002 [00:00<?, ? examples/s]

Map:   0%|          | 0/14501 [00:00<?, ? examples/s]

preprocessed: train={len(train_tok)}, test={len(test_tok)}


### Создание trainer'ов

Используется `FocalLossTrainer`.
Для каждого варианта текста создаётся отдельный trainer.

In [6]:
model_cfg = {
    'learning_rate': 2e-5,
    'epochs': 7,
    'batch_size': 32,
    'fp16': True,
    'focal_alpha': 0.25,
    'focal_gamma': 2.0,
}

trainers = {}
for name in tokenized:
    output_dir = f'finetuned_rubert_tiny_{name}'

    training_args = get_training_args(
        output_dir=output_dir,
        learning_rate=model_cfg['learning_rate'],
        num_train_epochs=model_cfg['epochs'],
        per_device_train_batch_size=model_cfg['batch_size'],
        per_device_eval_batch_size=model_cfg['batch_size'],
        max_length=model_config['max_length'],
        fp16=model_cfg['fp16'],
        gradient_checkpointing=model_config['gradient_checkpointing'],
    )

    trainer = FocalLossTrainer(
        focal_alpha=model_cfg['focal_alpha'],
        focal_gamma=model_cfg['focal_gamma'],
        model=model,
        args=training_args,
        train_dataset=tokenized[name]['train'],
        eval_dataset=tokenized[name]['test'],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainers[name] = trainer

### Запуск дообучения

Обучение выполняется на GPU (если доступно).

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')
model.to(device)

Устройство: cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(29564, 312, padding_idx=0)
      (position_embeddings): Embedding(512, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), e

#### На исходных текстах (raw)

In [8]:
trainers['raw'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.001930,0.991706,0.978888,0.990388,0.967652
2,No log,0.001554,0.992881,0.982109,0.980916,0.983304
3,No log,0.001549,0.993987,0.984793,0.989810,0.979826
4,No log,0.001528,0.993849,0.984487,0.986723,0.982261
5,No log,0.001589,0.994609,0.986368,0.991219,0.981565
6,No log,0.001681,0.993987,0.984830,0.987413,0.982261
7,No log,0.001725,0.994264,0.985512,0.989138,0.981913


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12663, training_loss=0.002227887133517217, metrics={'train_runtime': 464.6267, 'train_samples_per_second': 871.906, 'train_steps_per_second': 27.254, 'total_flos': 1493686181127168.0, 'train_loss': 0.002227887133517217, 'epoch': 7.0})

In [9]:
trainers['raw'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.001589,7,0.994609,0.986368,0.991219,0.981565


{'eval_loss': 0.0015892998781055212,
 'eval_accuracy': 0.9946091644204852,
 'eval_f1': 0.9863684026564138,
 'eval_precision': 0.9912188268352652,
 'eval_recall': 0.9815652173913043}

Сохранение модели.

In [10]:
model.save_pretrained(MODELS_DIR / 'finetuned_rubert_tiny_raw')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rubert_tiny_raw')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rubert_tiny_raw/tokenizer_config.json',
 '/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rubert_tiny_raw/tokenizer.json')

#### На нормализованных текстах (normalized)

In [11]:
trainers['normalized'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.001084,0.995577,0.988514,0.994942,0.982168
2,No log,0.001214,0.995508,0.988453,0.984779,0.992154
3,No log,0.001329,0.995716,0.988956,0.987900,0.990014
4,No log,0.001411,0.996130,0.990025,0.988968,0.991084
5,No log,0.001524,0.996199,0.990198,0.989669,0.990728
6,No log,0.001656,0.996683,0.991419,0.993907,0.988944
7,No log,0.001644,0.996752,0.991606,0.993202,0.990014


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12663, training_loss=0.00040752044421071495, metrics={'train_runtime': 465.979, 'train_samples_per_second': 869.541, 'train_steps_per_second': 27.175, 'total_flos': 1493970088095744.0, 'train_loss': 0.00040752044421071495, 'epoch': 7.0})

In [12]:
trainers['normalized'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.001644,7,0.996752,0.991606,0.993202,0.990014


{'eval_loss': 0.0016439235769212246,
 'eval_accuracy': 0.9967521249395342,
 'eval_f1': 0.9916056438649758,
 'eval_precision': 0.9932021466905188,
 'eval_recall': 0.9900142653352354}

Сохранение модели.

In [13]:
model.save_pretrained(MODELS_DIR / 'finetuned_rubert_tiny_norm')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rubert_tiny_norm')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rubert_tiny_norm/tokenizer_config.json',
 '/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rubert_tiny_norm/tokenizer.json')

#### На предобработанных текстах (preprocessed)

In [14]:
trainers['preprocessed'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.001298,0.993931,0.984369,0.994259,0.974675
2,No log,0.001364,0.993173,0.982750,0.973757,0.991910
3,No log,0.001069,0.995449,0.988327,0.993952,0.982765
4,No log,0.001132,0.995449,0.988331,0.993601,0.983116
5,No log,0.001193,0.995862,0.989429,0.991175,0.987689
6,No log,0.001249,0.995862,0.989418,0.992218,0.986634
7,No log,0.001227,0.995931,0.989604,0.991525,0.987689


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12691, training_loss=0.000745845605611106, metrics={'train_runtime': 465.9556, 'train_samples_per_second': 871.358, 'train_steps_per_second': 27.236, 'total_flos': 1497015635576832.0, 'train_loss': 0.000745845605611106, 'epoch': 7.0})

In [15]:
trainers['preprocessed'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.001227,7,0.995931,0.989604,0.991525,0.987689


{'eval_loss': 0.0012273824540898204,
 'eval_accuracy': 0.9959313150817185,
 'eval_f1': 0.9896035242290749,
 'eval_precision': 0.9915254237288136,
 'eval_recall': 0.9876890608512136}

Сохранение модели.

In [16]:
model.save_pretrained(MODELS_DIR / 'finetuned_rubert_tiny_p')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rubert_tiny_p')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rubert_tiny_p/tokenizer_config.json',
 '/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rubert_tiny_p/tokenizer.json')

## Замер CPU-инференса

Замеряется latency инференса на CPU для оценки пригодности модели к развёртыванию на сервере без GPU.

In [17]:
test_messages = [
    "Это честное сообщение от пользователя.",
    "🔥 Казино онлайн! Зарабатывай миллионы прямо сейчас! 💰💎",
    "Зарабатывай миллионы **онлайн** прямо сейчас!",
    "Работа на дому, легкий доход. Пиши в личку!",
    "Привет! Как дела? У меня всё отлично.",
    "steam gift 50$ - steamcommunity.com/gift-card/pay/50\n@everyone",
    "Давайте **вместе** будем писать про казино в чатах!!! Присоединяйтесь!",
    "Как же надоели эти сообщения про казино",
    "Добрый день. Для подачи документов необходимо пройти регистрацию здесь: stankin.ru",
    "3-4 часа и 8 тысяч твои!  Пиши  https://t.me/rasmuswork1",
]

sample_texts = test_messages * 10
suffix_map = {'raw': 'raw', 'normalized': 'norm', 'preprocessed': 'p'}
cpu_results = {}
for name, suffix in suffix_map.items():
    model_path = str(MODELS_DIR / f'finetuned_rubert_tiny_{suffix}')
    m = AutoModelForSequenceClassification.from_pretrained(model_path)
    tok = AutoTokenizer.from_pretrained(model_path)
    cpu_results[name] = benchmark_cpu_inference(m, tok, sample_texts, max_length=model_config['max_length'])
    print(f'{name}: {cpu_results[name]}')

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

raw: {'avg_ms_per_msg': 0.61, 'p95_ms_per_msg': 0.77, 'throughput_msgs_per_sec': 1646.58}


Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

normalized: {'avg_ms_per_msg': 0.56, 'p95_ms_per_msg': 0.59, 'throughput_msgs_per_sec': 1774.58}


Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

preprocessed: {'avg_ms_per_msg': 0.54, 'p95_ms_per_msg': 0.58, 'throughput_msgs_per_sec': 1848.44}
